# VRPP + Lookahead — interactive run

This notebook is only a **thin shell** over the `vrpp_lookahead` package.
All the logic (ORS matrix, instance, lookahead, Gurobi model, reports)
lives in `src/vrpp_lookahead/` — here you only pick the configuration and run it.

| Step | Function | Equivalent script |
|---|---|---|
| 1 | `build_ors_matrix(cfg)` | `scripts/01_build_ors_matrix.py` |
| 2 | `build_instance(cfg)` | `scripts/02_build_instance.py` |
| 3 | `run(cfg)` | `scripts/03_run_vrpp.py` |

The editable parameters (**B, Q, R, C, OMEGA, MIP_GAP, TIME_LIMIT, MAX_ROUTES**)
live in `config/instance_491_C7.yaml` and can be changed here in memory
(cell 3) without touching the file.

In [ ]:
# 1 — bootstrap: put the package on the path and load the configuration
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from _common import load_dotenv
load_dotenv(ROOT / '.env')          # ORS key (only needed in step 1)

from vrpp_lookahead import (Config, build_instance, build_ors_matrix, diagnose,
                            instance_summary, load_instance, run)

CFG = Config.from_yaml(ROOT / 'config' / 'instance_491_C7.yaml')
CFG.summary()

In [ ]:
# 2 — (optional) steps 1 and 2. Skip them if the matrix/instance is already on disk.
# build_ors_matrix(CFG)     # ~1 request per block of sources; needs ORS_API_KEY
# build_instance(CFG)       # joins attributes + coordinates + ORS matrix

INST = load_instance(CFG)
instance_summary(INST)

In [ ]:
# 3 — editable parameters (they override the YAML for this session only)
CFG.model.B = 16          # density [kg/m3]
CFG.model.Q = 3500        # vehicle capacity [kg]
CFG.model.R = 0.1625      # revenue [euro/kg]
CFG.model.C = 1.0         # cost [euro/km]
CFG.model.OMEGA = 0.1     # fixed cost per vehicle [euro]
CFG.model.MAX_ROUTES = 2  # k <= MAX_ROUTES
CFG.model.MIP_GAP = 0.05  # 5%
CFG.model.TIME_LIMIT = 21600
CFG.validate()

# If you changed B, reload the instance (CAP_CONT depends on B):
INST = load_instance(CFG)

DIAG = diagnose(INST, CFG)

In [ ]:
# 4 — run the VRPP + Lookahead (maps and Excel go to results/<label>/)
KPIS = run(CFG, diagnostics=DIAG)

In [ ]:
# 5 — summary
import pandas as pd
pd.DataFrame(KPIS).T